### boto3
#### kb chatbot, retrieval

In [1]:
import boto3
import json
import pandas as pd

# AWSセッション
session = boto3.Session(
    profile_name="bedrock-dev",
    region_name="ap-northeast-1"
)

# Bedrock Knowledge Bases Runtime
bedrock_agent_runtime = session.client(
    service_name="bedrock-agent-runtime"
)

# Knowledge Base ID
KB_ID = "B8DIZPAAO6"#"Your KB ID"

In [2]:
import os
import boto3

# AWSプロファイルを指定
os.environ["AWS_PROFILE"] = "bedrock-dev"
os.environ["AWS_DEFAULT_REGION"] = "ap-northeast-1"

# Session作成
session = boto3.Session()

# IAMユーザー確認
sts = session.client("sts")
identity = sts.get_caller_identity()

print("Profile:", session.profile_name)
print("ARN:", identity["Arn"])

Profile: bedrock-dev
ARN: arn:aws:iam::664418991639:user/bedrock-developer


In [3]:

query = "2026年10月18日の残課題を教えてください"

response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={
        "text": query
    },
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5
        }
    }
)

results = response["retrievalResults"]

print(f"取得件数: {len(results)}")

for i, result in enumerate(results, 1):
    print(f"\n--- 検索結果 {i} ---")
    print(result["content"].get("text", ""))
    print("スコア:", result.get("score"))


取得件数: 5

--- 検索結果 1 ---
今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特にプログラミング学習では、実際にコードを動かし、その仕組みや動作結果を説明できる状態を目指す。

### 残課題

**1. GraphRAGの改善方法を実装・検証する**

今回はWeb検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを比較しながら、品質への影響を検証したい。

**2.
スコア: 0.7306770086288452

--- 検索結果 2 ---
# 2026-10-18 日報：外出による気分転換と学習方法の整理

## 目的

外出して気分をリフレッシュし、休日の過ごし方を過ごした。

また、GraphRAGの品質改善方法について調査するとともに、今後の学習効率を高めるため、インプットとアウトプットの整理方法を明確にする。

## 実施内容・作業時間

| 内容 | 作業時間 | 実施内容 |
|---|---:|---|
| 外出 | 4時間30分 | 御誕生寺、越前たけふ駅、お多福食堂、西山公園を訪問し、ゴルフの打ちっぱなしを実施 |
| クラッシュ・ロワイヤル | 3時間30分 | ゲームをプレイ |
| VIVANTの視聴 | 1時間 | ドラマを視聴 |
スコア: 0.6768365502357483

--- 検索結果 3 ---
システム開発における品質管理**

テストプログラムやカバレッジ、ログ、pyproject.tomlについて調査した。テストによる動作確認、ログによるエラー調査、設定ファイルによる環境管理を組み合わせることで、保守・改善しやすい開発環境を整えることが重要である。

### 残課題

**1. クラウドサービスのコスト管理**

利用前に料金を概算し、利用中も日ごとのコストを確認する。ローカルのFAISSなどを利用する構成とも比較し、学習用途に適した構成を選択したい。

**2. 実データを用いたRAGの品質検証**

画像からの情報抽出、検索結果、生成回答の妥当性を確認する。
スコア: 0.5810970664024353

--

In [4]:

for i, result in enumerate(results, 1):
    print(f"\n===== 検索結果 {i} =====")

    metadata = result.get("metadata", {})

    if metadata:
        for key, value in metadata.items():
            print(f"{key}: {value}")
    else:
        print("metadataはありません")



===== 検索結果 1 =====
date: 2026-10-18
_source_uri: https://amzn-s3-bedrock-demo01.s3.ap-northeast-1.amazonaws.com/markdown/2026-10-18.md
date_confidence: high
_chunk_id: j-S6_ml8poQ5nzxfCtMBE43aTjbDCzWxe67imOYV6YU
_data_source_type: S3
_language_code: ja
_created_at: 2026-11-13T00:35:54Z
_data_source_id: VJPJPTY9NM
_file_type: PLAIN_TEXT
_document_title: 2026-10-18.md
category_confidence: middle
_last_updated_at: 2026-11-13T00:35:54Z
category: 振り返り

===== 検索結果 2 =====
date: 2026-10-18
_source_uri: https://amzn-s3-bedrock-demo01.s3.ap-northeast-1.amazonaws.com/markdown/2026-10-18.md
date_confidence: high
_chunk_id: nP4IsDy8jBqbHqny_aviLLxg6BDR0FPNhFDJo4Kn3gg
_data_source_type: S3
_language_code: ja
_created_at: 2026-11-13T00:35:54Z
_data_source_id: VJPJPTY9NM
_file_type: PLAIN_TEXT
_document_title: 2026-10-18.md
category_confidence: middle
_last_updated_at: 2026-11-13T00:35:54Z
category: 振り返り

===== 検索結果 3 =====
date: 2026-10-19
_source_uri: https://amzn-s3-bedrock-demo01.s3.ap-northeast

In [5]:

import pandas as pd

rows = []

for i, result in enumerate(results, 1):
    row = {
        "順位": i,
        "スコア": result.get("score"),
        "本文": result.get("content", {}).get("text", ""),
        "参照元": result.get("location", {})
    }

    # metadataを列として追加
    row.update(result.get("metadata", {}))

    rows.append(row)

df = pd.DataFrame(rows)

display(df)


,順位,スコア,本文,参照元,date,_source_uri,date_confidence,_chunk_id,_data_source_type,_language_code,_created_at,_data_source_id,_file_type,_document_title,category_confidence,_last_updated_at,category
0,1,0.730677,今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特...,{'s3Location': {'uri': 'https://amzn-s3-bedroc...,2026-10-18,https://amzn-s3-bedrock-demo01.s3.ap-northeast...,high,j-S6_ml8poQ5nzxfCtMBE43aTjbDCzWxe67imOYV6YU,S3,ja,2026-11-13T00:35:54Z,VJPJPTY9NM,PLAIN_TEXT,2026-10-18.md,middle,2026-11-13T00:35:54Z,振り返り
1,2,0.676837,# 2026-10-18 日報：外出による気分転換と学習方法の整理\n\n## 目的\n\n...,{'s3Location': {'uri': 'https://amzn-s3-bedroc...,2026-10-18,https://amzn-s3-bedrock-demo01.s3.ap-northeast...,high,nP4IsDy8jBqbHqny_aviLLxg6BDR0FPNhFDJo4Kn3gg,S3,ja,2026-11-13T00:35:54Z,VJPJPTY9NM,PLAIN_TEXT,2026-10-18.md,middle,2026-11-13T00:35:54Z,振り返り
2,3,0.581097,システム開発における品質管理**\n\nテストプログラムやカバレッジ、ログ、pyprojec...,{'s3Location': {'uri': 'https://amzn-s3-bedroc...,2026-10-19,https://amzn-s3-bedrock-demo01.s3.ap-northeast...,high,aKGvsDU3NDZE5oxG2Jn4t1__AmINz46P-j1pXElYPNw,S3,ja,2026-11-13T00:35:55Z,VJPJPTY9NM,PLAIN_TEXT,2026-10-19.md,middle,2026-11-13T00:35:55Z,クラウド
3,4,0.560073,- 複数の質問を入力し、それぞれの回答を取得できることを確認する\n- 質問に対する回答内容...,{'s3Location': {'uri': 'https://amzn-s3-bedroc...,2026-10-18,https://amzn-s3-bedrock-demo01.s3.ap-northeast...,high,_3LjGdzwRiK7c_gjTVKkE2meEFE4APcRiHCjm9e-Akk,S3,ja,2026-11-13T00:35:54Z,VJPJPTY9NM,PLAIN_TEXT,2026-10-18.md,middle,2026-11-13T00:35:54Z,振り返り
4,5,0.541211,Reactは便利な機能を提供する一方、基礎知識が不足していると、その利点やプログラムの構成を...,{'s3Location': {'uri': 'https://amzn-s3-bedroc...,2026-10-17,https://amzn-s3-bedrock-demo01.s3.ap-northeast...,high,M-xBT-ZuLb4GPetx35YRr2zbhraS3UGTXYsuNJTdB8w,S3,ja,2026-11-13T00:35:55Z,VJPJPTY9NM,PLAIN_TEXT,2026-10-17.md,middle,2026-11-13T00:35:55Z,振り返り


#### metadata filter retrieval

In [6]:
query = "残課題を教えてください"

response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={
        "text": query
    },
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5,
            "filter": {
                "equals": {
                    "key": "date",
                    "value": "2026-10-18"
                }
            }
        }
    }
)

results = response["retrievalResults"]

print(f"取得件数: {len(results)}")

for i, result in enumerate(results, 1):
    print(f"\n--- 検索結果 {i} ---")
    print(result["content"].get("text", ""))
    print("スコア:", result.get("score"))

取得件数: 5

--- 検索結果 1 ---
今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特にプログラミング学習では、実際にコードを動かし、その仕組みや動作結果を説明できる状態を目指す。

### 残課題

**1. GraphRAGの改善方法を実装・検証する**

今回はWeb検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを比較しながら、品質への影響を検証したい。

**2.
スコア: 0.7238121628761292

--- 検索結果 2 ---
**2. インプットとアウトプットを組み合わせた学習を実践する**

インプットとアウトプットの役割は整理できたが、実際の学習で十分に活用できているかは、今後確認する必要がある。学習した内容について、メモやコードリーディングで仕組みを理解した後、プログラムの実装やMarkdownでの整理を行い、理解と成果の両方を確認したい。


## 次回に向けて

**1. テストプログラムのハンズオン**

Pythonでテストプログラムを作成し、実際に実行しながらテストの基本的な仕組みを理解する。

- 正常系・異常系のテストケースを作成する
- `assert`を使用して、期待する結果と実際の結果を比較する
スコア: 0.2197287529706955

--- 検索結果 3 ---
# 2026-10-18 日報：外出による気分転換と学習方法の整理

## 目的

外出して気分をリフレッシュし、休日の過ごし方を過ごした。

また、GraphRAGの品質改善方法について調査するとともに、今後の学習効率を高めるため、インプットとアウトプットの整理方法を明確にする。

## 実施内容・作業時間

| 内容 | 作業時間 | 実施内容 |
|---|---:|---|
| 外出 | 4時間30分 | 御誕生寺、越前たけふ駅、お多福食堂、西山公園を訪問し、ゴルフの打ちっぱなしを実施 |
| クラッシュ・ロワイヤル | 3時間30分 | ゲームをプレイ |
| VIVANTの視聴 | 1時間 | ドラマを視聴 |
スコア: 0.20332047

#### retrieval(kb) + generation(myself), not metadata filter

In [7]:
# settings
MODEL_ID = "jp.anthropic.claude-haiku-4-5-20251001-v1:0"

bedrock_agent_runtime = boto3.client(
    "bedrock-agent-runtime",
    region_name="ap-northeast-1"
)

bedrock_runtime = boto3.client(
    "bedrock-runtime",
    region_name="ap-northeast-1"
)

# question
query = "2026年10月17日の残課題を教えて"

# retrieval
response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={
        "text": query
    },
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5
        }
    }
)

results = response["retrievalResults"]
print(f"取得件数: {len(results)}")

# retrieval results changes context
context = "\n\n".join(
    result["content"].get("text", "")
    for result in results
)

# making prompt
prompt = f"""
あなたは情報を整理するアシスタントです。

以下の参考情報をもとに質問に回答してください。

【参考情報】
{context}

【質問】
{query}

【回答ルール】
・参考情報に記載されている内容を根拠に回答する。
・参考情報にない情報は推測しない。
・情報が不足している場合は、その旨を伝える。
・日本語で回答する。
"""

# LLM answer generation
generation_response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ],
    inferenceConfig={
        "temperature": 0
    }
    
)

answer = generation_response["output"]["message"]["content"][0]["text"]

# result
print("===Answer===")
print(answer)

取得件数: 5
===Answer===
# 2026年10月17日の残課題

参考情報に基づいて、2026年10月17日の残課題は以下の通りです：

## 1. 学習内容の実践と理解の確認

- 複数の質問を入力し、それぞれの回答を取得できることを確認する
- 質問に対する回答内容が、参照元の文書と整合しているか確認する
- 回答の根拠となる参照情報や引用情報を取得できるか確認する

## 2. GraphRAGの改善方法を実装・検証する

- 調査した改善方法を実際のプログラムに適用し、効果を確認する
- エンティティ抽出やリレーション抽出の改善に取り組む
- 変更前後のナレッジグラフを比較しながら、品質への影響を検証する

## 3. 残課題（記載不完全）

参考情報に「**2.」の項目が途中で切れており、3番目以降の残課題の詳細が記載されていません。


#### retrieval(kb) + generation(kb), not metadata filter

In [8]:
import boto3

bedrock_agent_runtime = boto3.client(
    "bedrock-agent-runtime",
    region_name="ap-northeast-1"
)


query = "2026年10月18日の残課題を教えてください"

response = bedrock_agent_runtime.agentic_retrieve_stream(
    messages=[
        {
            "role": "user",
            "content": {
                "text": query
            }
        }
    ],
    retrievers=[
        {
            "configuration": {
                "knowledgeBase": {
                    "knowledgeBaseId": KB_ID
                }
            }
        }
    ],
    agenticRetrieveConfiguration={
        "foundationModelType": "MANAGED",
        "rerankingModelType": "MANAGED"
    },
    generateResponse=True
)

# イベントストリームから回答を取得
for event in response["stream"]:
    if "result" in event:
        result = event["result"]

        if "generatedResponse" in result:
            print("回答:")
            print(result["generatedResponse"]["answer"])

        print("\n参照元:")
        for item in result.get("results", []):
            print(item.get("content"))

回答:
2026年10月18日の日報における「残課題」は以下の2点です。

## 残課題

**1. GraphRAGの改善方法を実装・検証する**
今回はWeb検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを比較しながら、品質への影響を検証したい。

**2. インプットとアウトプットを組み合わせた学習を実践する**
インプットとアウトプットの役割は整理できたが、実際の学習で十分に活用できているかは、今後確認する必要がある。学習した内容について、メモやコードリーディングで仕組みを理解した後、プログラムの実装やMarkdownでの整理を行い、理解と成果の両方を確認したい。

参照元:
{'mimeType': 'text/plain', 'text': '今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特にプログラミング学習では、実際にコードを動かし、その仕組みや動作結果を説明できる状態を目指す。\n\n### 残課題\n\n**1. GraphRAGの改善方法を実装・検証する**\n\n今回はWeb検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを比較しながら、品質への影響を検証したい。\n\n**2.'}
{'mimeType': 'text/plain', 'text': '# 2026-10-18 日報：外出による気分転換と学習方法の整理\n\n## 目的\n\n外出して気分をリフレッシュし、休日の過ごし方を過ごした。\n\nまた、GraphRAGの品質改善方法について調査するとともに、今後の学習効率を高めるため、インプットとアウトプットの整理方法を明確にする。\n\n## 実施内容・作業時間\n\n| 内容 | 作業時間 | 実施内容 |\n|---|---:|---|\n| 外出 | 4時間30分 | 御誕生寺、越前たけふ駅、お多福食堂、西山公園を訪問し、ゴルフの打ちっぱなしを実施 |\n| クラッシュ・ロワイヤル | 3時間30分 | ゲ

#### retrieval(kb) + generation(kb), metadata filter

In [17]:
import boto3

bedrock_agent_runtime = boto3.client(
    "bedrock-agent-runtime",
    region_name="ap-northeast-1"
)


query = "10月19日の学びを教えて"

response = bedrock_agent_runtime.agentic_retrieve_stream(
    messages=[
        {
            "role": "user",
            "content": {
                "text": query
            }
        }
    ],
    retrievers=[
        {
            "configuration": {
                "knowledgeBase": {
                    "knowledgeBaseId": KB_ID,

                    # Metadata Filter
                    "retrievalOverrides": {
                        "filter": {
                            "equals": {
                                "key": "date",
                                "value": "2026-10-19"
                            }
                        },
                        "maxNumberOfResults": 5
                    }
                }
            }
        }
    ],
    agenticRetrieveConfiguration={
        "foundationModelType": "MANAGED",
        "rerankingModelType": "MANAGED"
    },
    generateResponse=True
)

# イベントストリームから回答を取得
for event in response["stream"]:
    if "result" in event:
        result = event["result"]

        if "generatedResponse" in result:
            print("回答:")
            print(result["generatedResponse"]["answer"])

        print("\n参照元:")
        for item in result.get("results", []):
            print(item.get("content"))

回答:
10月19日（2026年）の日報によると、以下の学びがありました。

## 実施内容
- **テストプログラムの基礎学習**（1時間30分）：テストの基礎やカバレッジ計測について学習
- **ログ管理の基礎学習**（30分）：ログの役割やエラー調査に必要な記録方法を調査
- **pyproject.tomlの調査**（30分）：Pythonプロジェクトの設定や依存関係の管理方法を調査
- **AI駆動開発の基礎学習**（30分）：動画視聴を通じてAI駆動開発の基本的な手順を調査
- **Azure AI Searchの料金調査**（30分）：料金体系やクラウドサービスのコスト管理について確認

## 主な学び
- **クラウドサービスのコスト管理**：Azure AI Searchの利用で3日間に約4,400円の費用が発生し、サービスの機能だけでなくリージョンや料金体系を事前に確認し、利用開始後も日ごとのコストを管理する必要があると学んだ。
- **画像を含むデータのRAGへの活用**：最近の生成AIを利用することで、手書きメモなど画像に含まれる情報も抽出して検索・回答生成に活用できることを確認した。
- **システム開発における品質管理**：テストプログラムやカバレッジ、ログ、pyproject.tomlについて調査し、テストによる動作確認、ログによるエラー調査、設定ファイルによる環境管理を組み合わせることで、保守・改善しやすい開発環境を整えることが重要だと学んだ。

## 残課題・次回に向けて
- クラウドサービスのコスト管理：利用前に料金を概算し、利用中も日ごとのコストを確認する。ローカルのFAISSなどとの構成比較も検討したい。
- 実データを用いたRAGの品質検証：画像からの情報抽出、検索結果、生成回答の妥当性を確認する。
- Azure AI SearchとローカルのFAISSを比較し、費用と機能の両面から適切な構成を検討する。
- テスト、ログ、設定管理を実際のプログラムに適用する、AI駆動開発の手順（コード確認・テスト・修正）を実践するなどを予定している。
- また、日報作成の負担軽減のため、作成時間を10分以内に制限し、重要な学びと次の課題を簡潔に記録する運用を試したいとのことです。

参照元:
{'mimeType': 'text/pla

### langchain_aws
#### langchain_aws retriever(kb), answer(kb)

In [19]:
from langchain_aws.retrievers.bedrock import agentic_retrieve

question = "10月19日の学びを教えて"

response = agentic_retrieve(
    knowledge_base_id=KB_ID,
    query=question,
    region_name="ap-northeast-1",
    generate_response=True,
    number_of_results=5
)

# 回答
print("【Answer】")
print(response["generatedResponse"]["answer"])

# 取得したドキュメント
print("\n【Retrieved Results】")
for result in response.get("retrievedReferences", []):
    print(result)

【Answer】
10月19日の日報によると、主な学びは以下の3点です。

## 1. クラウドサービスのコスト管理
Azure AI Searchを3日間利用したところ約4,400円の費用が発生しました。サービスの機能面だけでなく、リージョンや料金体系を事前に確認し、利用開始後も日ごとのコストを管理する必要があると学びました。ローカルのFAISSなどを利用する構成とも比較し、学習用途に適した構成を選択することが残課題として挙げられています。

## 2. 画像を含むデータのRAGへの活用
最近の生成AIを利用することで、手書きのメモなど画像に含まれる情報も抽出し、検索・回答生成に活用できることを確認しました。今後は、情報抽出の精度や検索結果の妥当性の検証が課題です。

## 3. システム開発における品質管理
テストプログラム、カバレッジ、ログ、pyproject.toml（設定管理）について調査しました。テストによる動作確認、ログによるエラー調査、設定ファイルによる環境管理を組み合わせることで、保守・改善しやすい開発環境を整えることが重要だと学びました。

## 次回に向けての課題
- Azure AI SearchとローカルFAISSの費用・機能比較による適切な構成検討
- 実際の学習記録データを用いたRAGの情報抽出・検索・回答生成の品質検証
- テスト・ログ・設定管理の実プログラムへの適用
- コード確認・テスト・修正を含むAI駆動開発手順の実践

また、日報作成に負担を感じ始めているため、作成時間を10分以内に制限し、重要な学びと次の課題を簡潔に記録する運用を試みたいとも振り返っています。

【Retrieved Results】


#### langchain_aws retriever(kb), answer(myself)

In [14]:
from langchain_aws.retrievers import AmazonKnowledgeBasesRetriever

retriever = AmazonKnowledgeBasesRetriever(
    knowledge_base_id=KB_ID,
    region_name="ap-northeast-1",
    retrieval_config={
        "managedSearchConfiguration": {
            "numberOfResults": 5
        }
    }
)

question = "10月19日の学びを教えて"

docs = retriever.invoke(question)

print(f"取得件数: {len(docs)}")

for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

取得件数: 5

--- Document 1 ---
# 2026-10-19 日報：システム開発の基礎学習と実データを用いたRAGの実装・運用検証

## 目的

システム開発に必要なテスト、ログ管理、設定管理などの基礎知識を身につけ、品質を確認しながら開発を進めるための考え方を整理する。

また、Azure AI Searchの料金体系を確認し、実際の学習記録データを用いてRAGを実装することで、画像を含むデータを検索・回答生成に活用できるか検証する。

## 実施内容・作業時間

| 内容 | 作業時間 | 実施内容 |
|---|---:|---|
Metadata: {'documentId': 's3://amzn-s3-bedrock-demo01/markdown/2026-10-19.md', 'location': {'s3Location': {'uri': 'https://amzn-s3-bedrock-demo01.s3.ap-northeast-1.amazonaws.com/markdown/2026-10-19.md'}, 'type': 'S3'}, 'score': 0.7984994053840637, 'type': 'TEXT', 'source_metadata': {'date': '2026-10-19', '_source_uri': 'https://amzn-s3-bedrock-demo01.s3.ap-northeast-1.amazonaws.com/markdown/2026-10-19.md', 'date_confidence': 'high', '_chunk_id': 'dgp0WYWNmbEL_6xdeO5oOwFw-Y_P3X7oefoGgPXshJs', '_data_source_type': 'S3', '_language_code': 'ja', '_created_at': '2026-11-13T00:35:55Z', '_data_source_id': 'VJPJPTY9NM', '_file_type': 'PLAIN_TEXT', '_document_title': '2026-10-19.md', 'category_confidence': 'middle', 

In [11]:
context = "\n\n".join(
    [
        f"[Document {i}]\n{doc.page_content}"
        for i, doc in enumerate(docs, 1)
    ]
)

print(context)

[Document 1]
# 2026-10-19 日報：システム開発の基礎学習と実データを用いたRAGの実装・運用検証

## 目的

システム開発に必要なテスト、ログ管理、設定管理などの基礎知識を身につけ、品質を確認しながら開発を進めるための考え方を整理する。

また、Azure AI Searchの料金体系を確認し、実際の学習記録データを用いてRAGを実装することで、画像を含むデータを検索・回答生成に活用できるか検証する。

## 実施内容・作業時間

| 内容 | 作業時間 | 実施内容 |
|---|---:|---|

[Document 2]
**5. 日報作成の効率化**

日報の作成に負担を感じ始めているため、作成時間を10分以内に制限し、重要な学びと次の課題を簡潔に記録する運用を試したい。

## 次回に向けて

- **Azure・コスト管理：** Azure AI SearchとローカルのFAISSを比較し、費用と機能の両面から適切な構成を検討する。
- **RAG：** 実際の学習記録データを用いて、情報抽出・検索・回答生成の品質を検証する。
- **システム開発：** テスト、ログ、設定管理を実際のプログラムに適用する。
- **AI駆動開発：** コードの確認・テスト・修正までを含めた開発手順を実践する。

[Document 3]
システム開発における品質管理**

テストプログラムやカバレッジ、ログ、pyproject.tomlについて調査した。テストによる動作確認、ログによるエラー調査、設定ファイルによる環境管理を組み合わせることで、保守・改善しやすい開発環境を整えることが重要である。

### 残課題

**1. クラウドサービスのコスト管理**

利用前に料金を概算し、利用中も日ごとのコストを確認する。ローカルのFAISSなどを利用する構成とも比較し、学習用途に適した構成を選択したい。

**2. 実データを用いたRAGの品質検証**

画像からの情報抽出、検索結果、生成回答の妥当性を確認する。

[Document 4]
クラウドサービスのコスト管理**

Azure AI Searchの利用で、3日間に約4,400円の費用が発生した。サービスの機能だけでなく、リージョンや料金体系を事前に確認し、利用開始後も日ごとのコストを

In [16]:
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate

# LLM設定
llm = ChatBedrockConverse(
    model="jp.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name="ap-northeast-1",
    temperature=0
)

# プロンプト
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
あなたは学習記録を整理するアシスタントです。

以下のContextに記載された情報だけを使用して回答してください。

ルール：
- Contextにない情報を推測して回答しない
- 情報が不足している場合は、その旨を伝える
- 学習内容をわかりやすく整理する
- 重要なポイントを箇条書きにする

Context:
{context}
"""
    ),
    (
        "human",
        "{question}"
    )
])

# Chain作成
chain = prompt | llm

# 回答生成
response = chain.invoke({
    "context": context,
    "question": question
})

print(response.content)

# 2026年10月19日の学習記録

## 主な学習内容

### 1. **システム開発の基礎知識**
- テストプログラムやカバレッジについて調査
- ログ管理の重要性を確認
- pyproject.tomlなどの設定ファイルによる環境管理を学習
- **ポイント：** テスト、ログ、設定管理を組み合わせることで、保守・改善しやすい開発環境が整えられる

### 2. **Azure AI Searchのコスト管理**
- 3日間の利用で約4,400円の費用が発生
- **学び：** サービス機能だけでなく、リージョンや料金体系を事前確認することが重要
- 利用開始後も日ごとのコストを管理する必要性を認識

### 3. **RAG（検索拡張生成）の実装検証**
- 画像を含むデータの活用可能性を確認
- 生成AIを利用して、手書きメモなどの画像情報も抽出・検索・回答生成に活用できることを検証

## 次のステップ

- Azure AI SearchとローカルのFAISSの比較検討
- 実データを用いたRAGの品質検証（情報抽出精度、検索結果の妥当性）
- システム開発の実践的な適用
